In [ ]:
!pip install -U pip setuptools wheel
!pip install jupyter pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost lightgbm imbalanced-learn 
!pip install kagglehub

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 14.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 9.8 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 9.3 MB/s  0:00:00 eta 0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 8.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 13.4 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━

In [2]:
#To download datset from Kaggle:
import kagglehub
from pathlib import Path
import pandas as pd
path = kagglehub.dataset_download("dartweichen/student-life") 
print("Path to dataset files:", path)

100%|██████████| 390M/390M [00:42<00:00, 9.56MB/s] 

Extracting files...


Path to dataset files: /Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1


In [12]:
from pathlib import Path

base_path = Path(path)
print(base_path)
print(base_path.exists())

/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1
True


In [13]:
for p in base_path.rglob("*Stress*"):
    print(p)

/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/survey/PerceivedStressScale.csv
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u44.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u13.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u05.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u52.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u09.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u25.json
/Users/leilatawfik/.

In [17]:
from pathlib import Path

stress_dir = base_path / "dataset" / "EMA" / "response" / "Stress"
print("stress_dir exists?", stress_dir.exists())
print("json count:", len(list(stress_dir.rglob("*.json"))))
print("csv count:", len(list(stress_dir.rglob("*.csv"))))

stress_dir exists? True
json count: 49
csv count: 0


In [18]:
import json

all_dfs = []
skipped = 0

for file in stress_dir.rglob("*.json"):
    try:
        text = file.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            skipped += 1
            continue

        try:
            obj = json.loads(text)
            df = pd.json_normalize(obj if isinstance(obj, list) else [obj])
        except json.JSONDecodeError:
            # JSON lines fallback
            rows = [json.loads(line) for line in text.splitlines() if line.strip()]
            df = pd.json_normalize(rows) if rows else pd.DataFrame()

        if df.empty:
            skipped += 1
            continue

        df["student_id"] = file.stem
        df["source_file"] = file.name
        all_dfs.append(df)

    except Exception:
        skipped += 1

stress_df = pd.concat(all_dfs, ignore_index=True, sort=False)

print("✅ stress_df shape:", stress_df.shape)
print("✅ participants:", stress_df["student_id"].nunique())
print("✅ skipped files:", skipped)
print("Columns:", list(stress_df.columns))
print(stress_df.head())

✅ stress_df shape: (2408, 6)
✅ participants: 49
✅ skipped files: 0
Columns: ['null', 'resp_time', 'level', 'location', 'student_id', 'source_file']
                       null   resp_time level location  student_id  \
0  43.70477575,-72.28844073  1364121982   NaN      NaN  Stress_u44   
1                         2  1364121983   NaN      NaN  Stress_u44   
2  43.70637091,-72.28704334  1364118696   NaN      NaN  Stress_u44   
3                         4  1364121980   NaN      NaN  Stress_u44   
4                         3  1364121985   NaN      NaN  Stress_u44   

       source_file  
0  Stress_u44.json  
1  Stress_u44.json  
2  Stress_u44.json  
3  Stress_u44.json  
4  Stress_u44.json  


In [22]:
stress_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2408 entries, 0 to 2407
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   null         240 non-null    str  
 1   resp_time    2408 non-null   int64
 2   level        2167 non-null   str  
 3   location     2167 non-null   str  
 4   student_id   2408 non-null   str  
 5   source_file  2408 non-null   str  
dtypes: int64(1), str(5)
memory usage: 113.0 KB


In [30]:
# Number of missing values in 'level'
missing_count = stress_df["level"].isna().sum()
total_count = len(stress_df["level"])
print(total_count)
print(missing_count/total_count * 100)

print("Missing values in 'level':", missing_count)

2408
10.008305647840531
Missing values in 'level': 241
